# A study of Resistance-conferring mutation patterns by FL-LPA in cases of MDR-TB Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² MDR-TB resistance mutation dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.s31m-1jxn/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# The metadata attribute is an object with attribute access
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.
We will enumerate all record sets in the dataset and display their fields and unique identifiers.

In [ ]:
# List all record sets and their field @id's

record_set_ids = []

print("Available record sets in this dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- Name: {record_set.name}\t@id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Name: {field.name}\t@id: {field.id}")
    print()

Let's preview a few records from the main record set, referencing it by its `@id`.


In [ ]:
# Display sample records from the main record set
main_record_set_id = record_set_ids[0]  # If only one record set, or pick appropriate by index
print(f"Showing first 2 records from record set @id: {main_record_set_id}")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if i >= 1:
        break

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames, referencing each by its `@id` for further analysis.

In [ ]:
# Extract all records from each record set into dataframes
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"Record set {rsid} loaded: shape = {dataframes[rsid].shape}")
    print(f"Columns: {list(dataframes[rsid].columns)}\n")

# Inspect top of the main record set
print("First 5 records of main record set:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's filter data and process numeric and categorical fields. We'll select a likely numeric field for demonstration (replace with actual field `@id` if found above), apply a threshold filter, normalize, and group/cross-tabulate if appropriate.

In [ ]:
# -- Update these IDs below with those discovered in the Overview outputs --
# Example: Suppose 'KM' (Kanamycin resistance) is a numeric/categorical field and 'sample_type' is a grouping factor.
# Get field IDs (for demonstration):

# List fields for main record set for reference
print("Main record set columns:", dataframes[main_record_set_id].columns.tolist())

# You may need to adapt field names based on printed outputs; for demo, we use 'KM' and 'sample_type'
numeric_field = 'KM'  # Replace with the @id or column name of a numeric field
group_field = 'sample_type'  # Replace with the @id or column name of a groupable field

if numeric_field in dataframes[main_record_set_id].columns:
    # Some columns may be categorical strings; try converting to numeric
    main_df = dataframes[main_record_set_id].copy()
    # Convert to numeric, coerce errors to NaN
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    # Filter out rows with missing numeric field
    filtered_df = main_df[main_df[numeric_field].notnull()]
    threshold = 0  # Example: Find samples where this field > 0
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by another field
    if group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].agg(['mean','count'])
        print(f"Grouped by {group_field}:")
        print(grouped)
else:
    print(f"Field '{numeric_field}' not found in columns. Please update 'numeric_field' to one of {dataframes[main_record_set_id].columns.tolist()}")

## 5. Visualization
Let's visualize the distribution of a selected field and its breakdown by sample type if possible.

In [ ]:
# Visualization example using matplotlib and seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in dataframes[main_record_set_id].columns:
    valid_df = dataframes[main_record_set_id].copy()
    valid_df[numeric_field] = pd.to_numeric(valid_df[numeric_field], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(valid_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot/groupplot if group field available
    if group_field in valid_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=valid_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded the FAIR² MDR-TB resistance pattern dataset using `mlcroissant`, explored its record sets and fields, and demonstrated downstream analysis and visualization referencing all entities by their `@id`. For more advanced or domain-specific analysis, use the schema metadata for precise field usage and follow dataset provenance best-practices.
